# StormScope-EU `true_color` forecast verification with `xskillscore`

Scores the nowcasts in `gtc-berlin/stormscope-eu_gtc_berlin_2026_EU_region` against what MTG-I1 FCI
actually observed, per lead time, and against a persistence baseline.

## Run on Coiled

The Arraylake bucket is in `eu-west-1`, so run the notebook VM there (CPU only, no GPU needed).
From the repo root, with `ARRAYLAKE_API_TOKEN` in `.env` (see `.env.example`):

```bash
uv sync --group notebook
set -a; source .env; set +a;
uv run --group notebook coiled notebook start --sync --name $USER-stormscope-verification \
    --workspace earthmover-devs --vm-type 'm6i.2xlarge' --region 'eu-west-1' \
    --open --idle-timeout '30 minutes' --allow-sync-from me \
    --env ARRAYLAKE_API_TOKEN=$ARRAYLAKE_API_TOKEN
```

`--sync` package-syncs the local uv env (incl. the `notebook` group: `xskillscore`, `dask`,
`matplotlib`) to the VM and mirrors the working directory, so this notebook shows up in JupyterLab.
Stop the VM when done: `coiled notebook stop $USER-stormscope-verification --workspace earthmover-devs`.
Do not `coiled run --name` this cluster: it shuts the VM down after the command.

## Method

- `lead_time = 0` is the observed FCI analysis frame of each init, so it is the verification
  truth: a forecast from init `t` at lead `L` is scored against the `lead_time = 0` frame of the
  init at `t + L`. Pairs whose valid time has no init (pipeline gaps) are dropped.
- Persistence baseline: the `lead_time = 0` frame of init `t` used as the forecast for `t + L`.
- `true_color` is uint8 RGB with fill value 0 (unwritten); 0 is masked to NaN. True colour is
  only meaningful in daylight, so pick a daytime window.
- Metrics per (init, lead) over `(y, x, rgb)`: RMSE, MAE, Pearson r. Skill score vs persistence:
  `1 - RMSE_model / RMSE_persistence`.
- Climatology baseline: shuffle the model forecasts along valid time, so each is scored against
  an unrelated observation; skill `1 - RMSE_model / RMSE_shuffled`.
- Change correlation: correlate the forecast change since init with the observed change, so the
  static scene (coasts, terminator, stationary cloud decks) no longer inflates the score.
  Persistence scores 0 by construction.
- Fractions skill score (FSS) of a brightness cloud mask over neighbourhood sizes: tolerant to
  small displacements, gives the scale from which the forecast is useful at each lead.

In [ ]:
import os

import arraylake
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import xskillscore as xs
from dotenv import load_dotenv
from scipy.ndimage import uniform_filter

load_dotenv()
client = arraylake.Client(token=os.environ.get("ARRAYLAKE_API_TOKEN"))  # None: use `arraylake auth login` credentials
repo = client.get_repo("gtc-berlin/stormscope-eu_gtc_berlin_2026_EU_region")
session = repo.readonly_session("main")
ds = xr.open_zarr(session.store, chunks={}, consolidated=False)
ds

## Choose a window

Each `(init, lead)` frame is ~8 MB uncompressed, so a full day of inits x 13 leads is a few GB.
Inits within 2 h before the window end lack later inits to verify their long leads; those pairs drop out.

In [ ]:
INITS = slice("2026-10-20T08:00", "2026-10-20T14:00")  # daytime, UTC
max_lead = ds.lead_time.values.max()

tc = ds.true_color.isel(ensemble=0)
tc = tc.where(tc > 0).astype("float32").chunk(y=-1, x=-1)  # 0 = fill value; xskillscore wants core dims in one chunk
fc = tc.sel(time=INITS)
t0, t1 = fc.time.values[[0, -1]]
obs = tc.isel(lead_time=0).sel(time=slice(t0, t1 + max_lead)).drop_vars(["lead_time", "valid_time"])
print(f"{fc.sizes['time']} inits, {obs.sizes['time']} observed frames")

## Pair forecasts with observations at valid time

Re-index each lead onto its valid time (`time + lead_time`) and inner-join with the observations.

In [ ]:
def at_valid_time(da, lead):
    return da.drop_vars(["valid_time", "lead_time"], errors="ignore").assign_coords(time=da.time + lead)


pairs = []
for lead in fc.lead_time.values:
    f, p, o = xr.align(
        at_valid_time(fc.sel(lead_time=lead), lead),
        at_valid_time(obs, lead),  # persistence: the frame observed at init
        obs,
        join="inner",
    )
    pairs.append(xr.Dataset({"model": f, "persistence": p, "obs": o}).assign_coords(lead_time=lead))
paired = xr.concat(pairs, "lead_time", join="outer")  # time is now the valid time
paired

## Scores per (valid time, lead)

In [ ]:
dims = ["y", "x", "rgb"]
scores = xr.Dataset(
    {
        f"{metric}_{name}": fn(paired[name], paired.obs, dim=dims, skipna=True)
        for metric, fn in (("rmse", xs.rmse), ("mae", xs.mae), ("r", xs.pearson_r))
        for name in ("model", "persistence")
    }
).compute()
scores

In [ ]:
mean = scores.mean("time")
mean["skill_rmse"] = 1 - mean.rmse_model / mean.rmse_persistence
lead_min = mean.lead_time / np.timedelta64(1, "m")

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
for ax, metric, label in zip(axes, ("rmse", "mae", "r"), ("RMSE (0-255)", "MAE (0-255)", "Pearson r"), strict=True):
    for name in ("model", "persistence"):
        ax.plot(lead_min, mean[f"{metric}_{name}"], marker="o", label=name)
    ax.set(xlabel="lead time [min]", title=label)
axes[0].legend()
fig.suptitle(f"true_color, inits {INITS.start} - {INITS.stop} UTC")
mean.skill_rmse.to_series().rename(lambda t: t / np.timedelta64(1, "m"))

## Climatology skill by shuffling forecasts

Pair each forecast with the observation of a random other valid time (same lead), repeated
`N_SHUFFLE` times. This keeps the forecast's own colour/cloud distribution but removes timing,
so it is the error of a forecast that knows only the climatology of the window.

In [ ]:
N_SHUFFLE = 5
rng = np.random.default_rng(0)
nt = paired.sizes["time"]
# ponytail: plain permutation can map a time to itself; use a derangement if nt is tiny
rmse_shuffled = xr.concat(
    [
        xs.rmse(
            paired.model.isel(time=rng.permutation(nt)).assign_coords(time=paired.time),
            paired.obs,
            dim=dims,
            skipna=True,
        )
        for _ in range(N_SHUFFLE)
    ],
    "iteration",
).compute()
mean["rmse_shuffled"] = rmse_shuffled.mean(["iteration", "time"])
mean["skill_rmse_clim"] = 1 - mean.rmse_model / mean.rmse_shuffled
mean[["skill_rmse", "skill_rmse_clim"]].to_dataframe().rename(index=lambda t: t / np.timedelta64(1, "m"))

## Correlation of changes since init

Spatial correlation of raw frames is dominated by what does not move. Correlating the change
`forecast - obs_at_init` with `obs - obs_at_init` asks only whether the model predicted what changed.

In [ ]:
init_frame = paired.persistence  # obs at init, already on valid time
r_change = xs.pearson_r(
    paired.model - init_frame, paired.obs - init_frame, dim=dims, skipna=True
).mean("time").compute()
r_change.to_series().rename(lambda t: t / np.timedelta64(1, "m")).rename("r_change")

## Fractions skill score (FSS)

Cloud mask = band-mean brightness > `THRESHOLD`. Fractions are the cloudy share in an `n x n`
pixel window (FCI pixel ~2 km here). FSS sums errors over all valid times before the ratio
(Roberts & Lean 2008). Useful skill: FSS > 0.5 + f/2, f = observed cloud fraction.

In [ ]:
THRESHOLD = 150  # 0-255; tune to separate cloud from surface in the window
SCALES = [1, 5, 11, 25, 51, 101]  # window size in pixels


def fractions(binary, n):
    return xr.apply_ufunc(
        lambda a: uniform_filter(a, size=(1,) * (a.ndim - 2) + (n, n), mode="constant"),
        binary.astype("float32"),
        input_core_dims=[["y", "x"]],
        output_core_dims=[["y", "x"]],
        dask="parallelized",
        output_dtypes=[np.float32],
    )


cloud = paired.mean("rgb") > THRESHOLD  # NaN -> False (clear)
fss = []
for n in SCALES:
    po = fractions(cloud.obs, n)
    for name in ("model", "persistence"):
        pf = fractions(cloud[name], n)
        num = ((pf - po) ** 2).sum(["time", "y", "x"])
        den = (pf**2 + po**2).sum(["time", "y", "x"])
        fss.append((1 - num / den).assign_coords(scale_km=2 * n, forecast=name))
fss = xr.concat(fss, "stack").set_index(stack=["scale_km", "forecast"]).unstack("stack").compute()
f = float(cloud.obs.mean())
print(f"observed cloud fraction {f:.2f}, useful FSS > {0.5 + f / 2:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, name in zip(axes, ("model", "persistence"), strict=True):
    for km in fss.scale_km.values:
        ax.plot(lead_min, fss.sel(forecast=name, scale_km=km), marker="o", label=f"{km} km")
    ax.axhline(0.5 + f / 2, color="k", ls="--", lw=1)
    ax.set(xlabel="lead time [min]", title=f"FSS {name}")
axes[0].legend(title="window")

## Where the model errs: RMSE map at the longest lead

Averaged over valid times and colour bands.

In [ ]:
last = paired.sel(lead_time=max_lead)
rmse_map = (((last.model - last.obs) ** 2).mean(["time", "rgb"]) ** 0.5).compute()  # plain xarray: no rechunk over time
rmse_map.plot(x="x", y="y", figsize=(12, 5), robust=True, cbar_kwargs={"label": "RMSE (0-255)"})
plt.title(f"RMSE at lead {max_lead / np.timedelta64(1, 'm'):.0f} min")